In [1]:
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 60.2 MB/s eta 0:00:00


In [2]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
import joblib
import pandas as pd

2025-09-11 19:35:22.378390: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1757619322.538620      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1757619322.583200      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
print("Loading and preparing data...")
train_df = pd.read_csv("/kaggle/input/map-charting-student-math-misunderstandings/train.csv")

# Create the same input text as in Phase 1
train_df['input_text'] = train_df.apply(
    lambda row: f"Question: {row.QuestionText} Answer: {row.MC_Answer} Explanation: {row.StudentExplanation}", 
    axis=1
)

Loading and preparing data...


In [4]:
train_df['Category'] = train_df['Category'].astype(str)
train_df['Misconception'] = train_df['Misconception'].astype(str)
train_df['full_label'] = train_df['Category'] + ':' + train_df['Misconception']
train_df['label_id'] = pd.Categorical(train_df['full_label']).codes

In [5]:
print("Loading fine-tuned sentence transformer model...")
model = SentenceTransformer('/kaggle/input/sberfine/sbert-finetuned-model')

Loading fine-tuned sentence transformer model...


In [6]:
print("Generating embeddings for the training data...")
train_embeddings = model.encode(train_df['input_text'].tolist(), convert_to_tensor=True, show_progress_bar=True)
train_embeddings_np = train_embeddings.cpu().numpy()

Generating embeddings for the training data...


Batches:   0%|          | 0/1147 [00:00<?, ?it/s]

In [7]:
print("Building FAISS index...")
embedding_dim = train_embeddings_np.shape[1]
# We use IndexFlatL2, which performs an exact search using L2 distance (Euclidean distance).
index = faiss.IndexFlatL2(embedding_dim)
index.add(train_embeddings_np) 

Building FAISS index...


In [8]:
print("Saving artifacts for submission...")
faiss.write_index(index, "train_faiss.index")
# We also need the original labels corresponding to each embedding.
joblib.dump(train_df['full_label'].tolist(), 'train_labels.joblib')


Saving artifacts for submission...


['train_labels.joblib']

In [9]:
def predict_knn(query_text, k=3):
    """
    Takes a new text, finds its k-nearest neighbors, and returns their labels.
    """
    # 1. Generate embedding for the new text
    query_embedding = model.encode([query_text])
    
    # 2. Search the FAISS index for the k nearest neighbors
    # D = distances, I = indices of the neighbors in the original training set
    D, I = index.search(query_embedding, k)
    
    # 3. Retrieve the labels of these neighbors
    neighbor_indices = I[0]
    
    # Load the saved labels
    train_labels = joblib.load('train_labels.joblib')
    
    # Aggregate predictions (e.g., by majority vote or just return the list)
    # For MAP@3, we care about the ranked list, so let's get the labels of the neighbors in order.
    predictions = [train_labels[i] for i in neighbor_indices]
    
    return predictions


In [10]:
test_query = "Question: What is 3 + 5? Answer: 7 Explanation: i added wrong"
predicted_labels = predict_knn(test_query, k=3)
print(f"\nTest Query: '{test_query}'")
print(f"Predicted Top 3 Labels (ranked): {predicted_labels}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Test Query: 'Question: What is 3 + 5? Answer: 7 Explanation: i added wrong'
Predicted Top 3 Labels (ranked): ['False_Neither:nan', 'True_Neither:nan', 'True_Neither:nan']
